# MathGuard ? Lightweight Verification of LLM Mathematical Reasoning

Default execution summarizes saved CSVs without model downloads or training. See README.md for methodology and limitations. Original cells and outputs are preserved locally in `local_archive/Math_Repair.ipynb`.


In [ ]:
from pathlib import Path
from datetime import datetime
import json

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data" / "train.json").exists(), "Start Jupyter from the repository or notebooks directory."
DATA = ROOT / "data"
RECORDED = ROOT / "results"
# Default: inspect recorded results only. Explicitly enable expensive GPU experiments.
RUN_EXPERIMENTS = False
if RUN_EXPERIMENTS:
    import torch, random, gc, re
    import pandas as pd
    from IPython.display import display
    random.seed(42)
    torch.manual_seed(42)
    assert torch.cuda.is_available(), "GPU experiments require CUDA."
    RUN = ROOT / "runs" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
    RUN.mkdir(parents=True, exist_ok=False)
    print("New experiment outputs:", RUN)


## Load preserved datasets


In [ ]:
if RUN_EXPERIMENTS:
    train_data = json.loads((DATA / "train.json").read_text())
    test_data = json.loads((DATA / "test.json").read_text())
    assert len(train_data) == 240 and len(test_data) == 48


## Quantized main model and label parser


In [ ]:
if RUN_EXPERIMENTS:
    # Qwen
    
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    
    MAIN_ID = "Qwen/Qwen3-4B-Instruct-2507"
    
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    
    main_tok = AutoTokenizer.from_pretrained(MAIN_ID)
    
    main_model = AutoModelForCausalLM.from_pretrained(
        MAIN_ID,
        quantization_config=bnb,
        device_map="auto",
        dtype=torch.float16
    ).eval()
    
    @torch.inference_mode()
    def main_generate(prompt, max_tokens=110):
        messages = [
            {"role": "system", "content": "You evaluate mathematical claims carefully."},
            {"role": "user", "content": prompt}
        ]
    
        inputs = main_tok.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True
        ).to(main_model.device)
    
        outputs = main_model.generate(
            inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=main_tok.eos_token_id
        )
    
        return main_tok.decode(
            outputs[0][inputs.shape[1]:],
            skip_special_tokens=True
        ).strip()
    
    def get_label(response):
        match = re.search(r"\b(VALID|INVALID)\b", response.upper())
        return match.group(1) if match else "UNKNOWN"
    
    print(main_generate(
        "Is this claim correct? sqrt((-3)^2) = -3. "
        "Answer VALID or INVALID, then explain briefly."
    ))


## Baseline evaluation


In [ ]:
if RUN_EXPERIMENTS:
    #baseline
    baseline = []
    
    for i, item in enumerate(test_data):
        prompt = (
            "Determine whether this mathematical claim is correct.\n"
            f"Claim: {item['problem']}\n"
            "Answer VALID or INVALID first, then briefly explain."
        )
    
        raw = main_generate(prompt)
        prediction = get_label(raw)
    
        baseline.append({
            **item,
            "prediction": prediction,
            "raw_response": raw,
            "correct": prediction == item["label"]
        })
    
        if (i + 1) % 12 == 0:
            print(f"Evaluated {i+1}/48")
    
    baseline_df = pd.DataFrame(baseline)
    baseline_df.to_csv(RUN / "baseline.csv", index=False)
    
    print("Baseline accuracy:", round(baseline_df.correct.mean()*100, 2), "%")
    
    display(
        baseline_df.loc[
            ~baseline_df.correct,
            ["problem", "label", "prediction", "raw_response"]
        ].head(10)
    )


## Training imports


In [ ]:
if RUN_EXPERIMENTS:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from datasets import Dataset
    from peft import LoraConfig
    from trl import SFTTrainer, SFTConfig
    import torch, gc, json
    
    train_data = json.loads((DATA / "train.json").read_text())


## Specialist training examples


In [ ]:
if RUN_EXPERIMENTS:
    # SLM training
    if "main_model" in globals():
        del main_model
    
    gc.collect()
    torch.cuda.empty_cache()
    
    from datasets import Dataset
    from peft import LoraConfig
    from trl import SFTTrainer, SFTConfig
    
    SLM_ID = "Qwen/Qwen2.5-0.5B-Instruct"
    slm_tok = AutoTokenizer.from_pretrained(SLM_ID)
    
    def make_training_row(item):
        return {
            "prompt": [
                {
                    "role": "system",
                    "content": "You are a mathematical assumption checker."
                },
                {
                    "role": "user",
                    "content": (
                        f"Check this claim: {item['problem']}\n"
                        "Answer VALID or INVALID, then explain."
                    )
                }
            ],
            "completion": [
                {
                    "role": "assistant",
                    "content": f"{item['label']}. {item['reason']}"
                }
            ]
        }
    
    train_ds = Dataset.from_list([
        make_training_row(x) for x in train_data
    ])
    
    print("Training examples:", len(train_ds))


## LoRA fine-tuning


In [ ]:
if RUN_EXPERIMENTS:
    # LoRA fine-tuning
    
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "v_proj"]
    )
    
    train_config = SFTConfig(
        output_dir=str(RUN / "training"),
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        max_length=256,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        fp16=True,
        completion_only_loss=True
    )
    
    trainer = SFTTrainer(
        model=SLM_ID,
        args=train_config,
        train_dataset=train_ds,
        peft_config=lora_config,
        processing_class=slm_tok
    )
    
    trainer.train()
    
    trainer.save_model(str(RUN / "math_slm_lora"))
    slm_tok.save_pretrained(str(RUN / "math_slm_lora"))
    
    print("LoRA training complete")


## Load test examples


In [ ]:
if RUN_EXPERIMENTS:
    import json
    import re
    import pandas as pd
    
    test_data = json.loads((DATA / "test.json").read_text())
    
    def get_label(response):
        match = re.search(r"\b(VALID|INVALID)\b", response.upper())
        return match.group(1) if match else "UNKNOWN"
    
    print("Loaded", len(test_data), "test examples")


## Specialist evaluation


In [ ]:
if RUN_EXPERIMENTS:
    #Test
    slm_model = trainer.model.eval()
    
    @torch.inference_mode()
    def slm_check(claim):
        messages = [
            {"role": "system", "content": "You are a mathematical assumption checker."},
            {"role": "user", "content": (
                f"Check this claim: {claim}\n"
                "Answer VALID or INVALID, then explain."
            )}
        ]
    
        inputs = slm_tok.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True
        ).to(slm_model.device)
    
        outputs = slm_model.generate(
            inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=slm_tok.eos_token_id
        )
    
        return slm_tok.decode(
            outputs[0][inputs.shape[1]:],
            skip_special_tokens=True
        ).strip()
    
    slm_results = []
    
    for item in test_data:
        response = slm_check(item["problem"])
        pred = get_label(response)
    
        slm_results.append({
            **item,
            "slm_prediction": pred,
            "slm_feedback": response,
            "slm_correct": pred == item["label"]
        })
    
    slm_df = pd.DataFrame(slm_results)
    slm_df.to_csv(RUN / "slm_results.csv", index=False)
    
    print("Trained SLM accuracy:",
          round(slm_df.slm_correct.mean()*100, 2), "%")


## Load predictions for correction


In [ ]:
if RUN_EXPERIMENTS:
    baseline = pd.read_csv(RUN / "baseline.csv").to_dict("records")
    slm_results = pd.read_csv(RUN / "slm_results.csv").to_dict("records")


## Main model with specialist feedback


In [ ]:
if RUN_EXPERIMENTS:
    # correction ( LLM + SLM)
    
    if "trainer" in globals():
        del trainer
    
    if "slm_model" in globals():
        del slm_model
    
    gc.collect()
    torch.cuda.empty_cache()
    
    main_model = AutoModelForCausalLM.from_pretrained(
        MAIN_ID,
        quantization_config=bnb,
        device_map="auto",
        dtype=torch.float16
    ).eval()
    
    combined = []
    
    for item, base, critic in zip(test_data, baseline, slm_results):
        prompt = f"""
    Mathematical claim:
    {item['problem']}
    
    Your initial judgment:
    {base['raw_response']}
    
    A smaller specialist model checked the claim:
    {critic['slm_feedback']}
    
    The specialist may be wrong. Independently verify the
    mathematics before making your final decision.
    
    Answer VALID or INVALID first, then explain briefly.
    """
    
        response = main_generate(prompt)
        pred = get_label(response)
    
        combined.append({
            "problem": item["problem"],
            "gold": item["label"],
            "baseline_correct": base["correct"],
            "slm_correct": critic["slm_correct"],
            "combined_correct": pred == item["label"],
            "final_prediction": pred,
            "final_response": response
        })
    
    combined_df = pd.DataFrame(combined)
    combined_df.to_csv(RUN / "combined_results.csv", index=False)
    
    print("Combined evaluation complete")


## Load new experiment results


In [ ]:
if RUN_EXPERIMENTS:
    import pandas as pd
    from pathlib import Path
    
    
    
    baseline_df = pd.read_csv(RUN / "baseline.csv")
    slm_df = pd.read_csv(RUN / "slm_results.csv")
    combined_df = pd.read_csv(RUN / "combined_results.csv")
    
    for col in ["correct"]:
        baseline_df[col] = baseline_df[col].astype(bool)
    
    slm_df["slm_correct"] = slm_df["slm_correct"].astype(bool)
    
    for col in ["baseline_correct", "slm_correct", "combined_correct"]:
        combined_df[col] = combined_df[col].astype(bool)
    
    print("Baseline:", len(baseline_df))
    print("SLM:", len(slm_df))
    print("Combined:", len(combined_df))
    print("All results loaded successfully!")


## Standard test comparison


In [ ]:
if RUN_EXPERIMENTS:
    # Final Comparison
    print("Qwen3-4B alone:",
          round(baseline_df.correct.mean()*100, 2), "%")
    
    print("LoRA-trained SLM:",
          round(slm_df.slm_correct.mean()*100, 2), "%")
    
    print("Qwen3-4B + SLM:",
          round(combined_df.combined_correct.mean()*100, 2), "%")
    
    print("\nImproved cases:")
    display(combined_df[
        (~combined_df.baseline_correct) &
        combined_df.combined_correct
    ][["problem", "gold", "final_prediction"]])
    
    print("\nRegressed cases:")
    display(combined_df[
        combined_df.baseline_correct &
        (~combined_df.combined_correct)
    ][["problem", "gold", "final_prediction"]])


## Load preserved hard cases


In [ ]:
if RUN_EXPERIMENTS:
    hard_test = json.loads((DATA / "hard_test.json").read_text())
    assert len(hard_test) == 24


## Hard baseline


In [ ]:
if RUN_EXPERIMENTS:
    # ind LLM
    
    hard_baseline = []
    
    for item in hard_test:
        prompt = (
            f"Determine whether this mathematical claim is correct:\n"
            f"{item['problem']}\n"
            "Answer VALID or INVALID first, then briefly explain."
        )
    
        response = main_generate(prompt)
        prediction = get_label(response)
    
        hard_baseline.append({
            **item,
            "prediction": prediction,
            "raw_response": response,
            "correct": prediction == item["label"]
        })
    
    hard_baseline_df = pd.DataFrame(hard_baseline)
    hard_baseline_df.to_csv(RUN / "hard_baseline.csv", index=False)
    
    print("LLM baseline accuracy:",
          round(hard_baseline_df.correct.mean()*100, 2), "%")


## Reload trained specialist and evaluate hard cases


In [ ]:
if RUN_EXPERIMENTS:
    
    # loRA trained SLM
    
    import gc
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel
    
    if "main_model" in globals():
        del main_model
    
    gc.collect()
    torch.cuda.empty_cache()
    
    SLM_ID = "Qwen/Qwen2.5-0.5B-Instruct"
    
    slm_tok = AutoTokenizer.from_pretrained(SLM_ID)
    
    base_slm = AutoModelForCausalLM.from_pretrained(
        SLM_ID,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    
    slm_model = PeftModel.from_pretrained(
        base_slm,
        str(RUN / "math_slm_lora")
    ).eval()
    
    hard_slm = []
    
    for item in hard_test:
        response = slm_check(item["problem"])
        prediction = get_label(response)
    
        hard_slm.append({
            **item,
            "slm_feedback": response,
            "slm_prediction": prediction,
            "slm_correct": prediction == item["label"]
        })
    
    hard_slm_df = pd.DataFrame(hard_slm)
    hard_slm_df.to_csv(RUN / "hard_slm.csv", index=False)
    
    print("SLM accuracy:",
          round(hard_slm_df.slm_correct.mean()*100, 2), "%")


## Hard combined evaluation


In [ ]:
if RUN_EXPERIMENTS:
    # Eval
    
    del slm_model, base_slm
    gc.collect()
    torch.cuda.empty_cache()
    
    from transformers import BitsAndBytesConfig
    
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    
    main_tok = AutoTokenizer.from_pretrained(MAIN_ID)
    
    main_model = AutoModelForCausalLM.from_pretrained(
        MAIN_ID,
        quantization_config=bnb,
        device_map="auto",
        dtype=torch.float16
    ).eval()
    
    hard_combined = []
    
    for item, base, critic in zip(hard_test, hard_baseline, hard_slm):
    
        prompt = f"""
    Mathematical claim:
    {item['problem']}
    
    Your earlier assessment:
    {base['raw_response']}
    
    A smaller mathematical specialist reports:
    {critic['slm_feedback']}
    
    Verify the mathematics independently.
    The specialist may be incorrect.
    
    Answer VALID or INVALID first, then explain briefly.
    """
    
        response = main_generate(prompt)
        prediction = get_label(response)
    
        hard_combined.append({
            "problem": item["problem"],
            "gold": item["label"],
            "baseline_correct": base["correct"],
            "slm_correct": critic["slm_correct"],
            "combined_correct": prediction == item["label"],
            "final_prediction": prediction,
            "final_response": response
        })
    
    hard_combined_df = pd.DataFrame(hard_combined)
    hard_combined_df.to_csv(RUN / "hard_combined.csv", index=False)
    
    print("Combined evaluation completed")


## Hard comparison


In [ ]:
if RUN_EXPERIMENTS:
    print("Qwen3-4B:", round(hard_baseline_df.correct.mean()*100, 2), "%")
    print("LoRA-trained SLM:", round(hard_slm_df.slm_correct.mean()*100, 2), "%")
    print("Qwen3-4B + SLM:", round(hard_combined_df.combined_correct.mean()*100, 2), "%")
    
    print("\nImproved cases:")
    display(hard_combined_df[
        (~hard_combined_df.baseline_correct) &
        hard_combined_df.combined_correct
    ])
    
    print("\nRegressed cases:")
    display(hard_combined_df[
        hard_combined_df.baseline_correct &
        (~hard_combined_df.combined_correct)
    ])


## Recorded results (read only)


In [ ]:
import csv
for filename, column in [
    ("baseline.csv", "correct"), ("slm_results.csv", "slm_correct"),
    ("combined_results.csv", "combined_correct"), ("hard_baseline.csv", "correct"),
    ("hard_slm.csv", "slm_correct"), ("hard_combined.csv", "combined_correct"),
]:
    with (RECORDED / filename).open(encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert all(row[column] in {"True", "False"} for row in rows)
    correct = sum(row[column] == "True" for row in rows)
    print(f"{filename}: {correct}/{len(rows)} ({100 * correct / len(rows):.2f}%)")
